In [ ]:
import ast
import glob
from collections import defaultdict
from datetime import date

import wandb
import numpy as np
import pandas as pd

# Set display options to show all columns and format floats nicely
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

user = "imos-buildingenergy-1"
ENTITY = "imos-buildingenergy-1"
PROJECT = "TabPHM-autoencoders-checks_02_10_night"

today = date.today()
api = wandb.Api()

# # Find all csv files in the current directory
csv_files = glob.glob("*.csv")
# # Collect all the names of the csv files without the extension
csv_names = [csv_file[:-4] for csv_file in csv_files]
project_name = "TabPHM-autoencoders-checks_02_10_night"
 

if project_name not in csv_names:
    runs = api.runs(f"{user}/{project_name}")

    summary_list, config_list, name_list = [], [], []
    for run in runs:
        # .summary contains the output keys/values for metrics like accuracy.
        #  We call ._json_dict to omit large files
        summary_list.append(run.summary._json_dict)

        # .config contains the hyperparameters.
        #  We remove special values that start with _.
        config_list.append(
            {k: v for k, v in run.config.items() if not k.startswith("_")}
        )

        # .name is the human-readable name of the run.
        name_list.append(run.name)

    runs_df = pd.DataFrame(
        {"summary": summary_list, "config": config_list, "name": name_list}
    )

    runs_df.to_csv(f"{project_name}.csv")
else:
    runs_df = pd.read_csv(f"{project_name}.csv", index_col=0)

    for row in runs_df.iloc:
        row["summary"] = ast.literal_eval(row["summary"])
        row["config"] = ast.literal_eval(row["config"])


for row in runs_df.iloc:
    row["summary"].update(row["config"])

lst = [i["summary"] for i in runs_df.iloc]
df = pd.DataFrame.from_dict(lst)

df_init = df.copy()

# Get average epoch run time
df["epoch_run_time"] = df["_runtime"] / df["epoch"]



### Go over different config groups and flatten the content from the nested dictionaries into separate columns

In [8]:
def normalize_column(df, column_to_normalize):
    # Use json_normalize to flatten the nested dictionaries into separate columns
    flattened_df = pd.json_normalize(df[column_to_normalize])
    # Rename columns to include 'nested_column' prefix
    flattened_df.columns = [
        f"{column_to_normalize}.{col}" for col in flattened_df.columns
    ]
    # Concatenate the flattened DataFrame with the original DataFrame
    result_df = pd.concat([df, flattened_df], axis=1)
    # Get new columns names
    new_columns = flattened_df.columns
    # Drop the original nested column if needed
    result_df.drop(column_to_normalize, axis=1, inplace=True)
    return result_df, new_columns


# Config columns to normalize
columns_to_normalize = ["model", "dataset", "callbacks", "paths"]

# Keep track of config columns added
config_columns = []
for column in columns_to_normalize:
    df, columns = normalize_column(df, column)
    config_columns.extend(columns)

In [ ]:


# Describe only the numerical columns
print("--- 📊 Numerical Columns Summary ---")
display(df.describe(include='number'))

# Describe only the object/categorical columns
print("\n--- 📝 Object/Categorical Columns Summary ---")
display(df.describe(include=['object']))



--- 📊 Numerical Columns Summary ---


/home/levtel/miniconda3/envs/phm2/lib/python3.12/site-packages/pandas/core/nanops.py:1016: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)


,_runtime,_step,_timestamp,epoch,lr-Adam,train/loss_step,trainer/global_step,seed,num_workers,val_batch_size,prefetch_factor,test_batch_size,train_batch_size,model/params/total,model/params/trainable,AvgTime/train_batch_mean,AvgTime/train_batch_std,AvgTime/train_epoch_mean,AvgTime/train_epoch_std,AvgTime/val_batch_mean,AvgTime/val_batch_std,AvgTime/val_epoch_mean,AvgTime/val_epoch_std,test/loss,test/mae,test/mse,test/rmse,train/loss_epoch,train/mae,train/mse,train/rmse,val/loss,val/mae,val/mse,val/rmse,epoch_run_time,model.latent_dim,model.dropout_prob,model.input_seq_len,model.input_channels,dataset.dataset_cfg.stride,dataset.dataset_cfg.seq_len,dataset.dataset_cfg.pred_len,dataset.dataset_cfg.label_len,callbacks.model_summary.max_depth,callbacks.early_stopping.patience,callbacks.early_stopping.min_delta,callbacks.model_checkpoint.save_top_k
count,54.00,54.00,54.00,54.00,54.00,54.00,54.00,54.00,54.00,54.00,54.00,54.00,54.00,54.00,54.00,48.00,48.00,48.00,48.00,48.00,48.00,48.00,48.00,48.00,48.00,48.00,48.00,50.00,50.00,50.00,50.00,50.00,50.00,50.00,50.00,54.00,54.00,54.00,54.00,54.00,54.00,54.00,54.00,54.00,54.00,54.00,54.00,54.00
mean,2048.47,2062.31,1759385863.91,18.07,0.00,2286.12,98532.67,42.00,4.00,986.67,10.00,986.67,433.04,755587.56,755587.56,0.02,0.00,84.50,3.52,0.01,0.00,2.95,0.11,1478.75,4.90,4436.25,6.56,1283402.15,16.28,3845266.82,245.63,1420.70,4.79,4261.00,6.46,inf,124.44,0.20,64.00,2.96,1.00,64.00,0.00,0.00,-1.00,25.00,0.00,1.00
std,2421.98,2536.71,14665.26,5.64,0.00,16324.21,127058.57,0.00,0.00,192.15,0.00,192.15,353.43,401473.83,401473.83,0.00,0.00,53.32,1.62,0.00,0.00,0.31,0.06,10134.90,29.49,30404.70,38.29,6388875.42,69.58,19167433.89,1117.15,9930.07,28.89,29790.36,37.51,NaN,18.30,0.00,0.00,0.19,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
min,20.00,21.00,1759357555.33,0.00,0.00,0.00,1049.00,42.00,4.00,16.00,10.00,16.00,16.00,183208.00,183208.00,0.02,0.00,28.31,0.74,0.01,0.00,2.46,0.03,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,30.20,32.00,0.20,64.00,2.00,1.00,64.00,0.00,0.00,-1.00,25.00,0.00,1.00
25%,699.50,544.00,1759375940.58,20.00,0.00,0.00,22120.00,42.00,4.00,1024.00,10.00,1024.00,128.00,380172.00,380172.00,0.02,0.00,41.32,2.49,0.01,0.00,2.69,0.05,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,51.41,128.00,0.20,64.00,3.00,1.00,64.00,0.00,0.00,-1.00,25.00,0.00,1.00
50%,1103.00,986.00,1759389995.93,20.00,0.00,0.00,44220.00,42.00,4.00,1024.00,10.00,1024.00,256.00,380442.00,380442.00,0.02,0.00,69.41,3.08,0.01,0.00,2.98,0.10,0.00,0.02,0.01,0.04,0.00,0.02,0.01,0.05,0.00,0.02,0.01,0.04,92.72,128.00,0.20,64.00,3.00,1.00,64.00,0.00,0.00,-1.00,25.00,0.00,1.00
75%,2998.00,3196.75,1759397213.50,20.00,0.01,0.07,154770.00,42.00,4.00,1024.00,10.00,1024.00,512.00,1167436.00,1167436.00,0.02,0.00,109.32,4.52,0.01,0.00,3.23,0.14,0.02,0.10,0.06,0.14,0.76,0.44,2.28,0.87,0.03,0.12,0.09,0.17,170.14,128.00,0.20,64.00,3.00,1.00,64.00,0.00,0.00,-1.00,25.00,0.00,1.00
max,12956.98,13888.00,1759414024.34,20.00,0.01,119997.78,692399.00,42.00,4.00,1024.00,10.00,1024.00,1024.00,1167436.00,1167436.00,0.02,0.01,172.91,7.29,0.01,0.01,3.48,0.27,70230.30,204.25,210690.83,265.01,36766636.00,359.69,110299888.00,6063.55,70230.30,204.25,210690.83,265.01,inf,128.00,0.20,64.00,3.00,1.00,64.00,0.00,0.00,-1.00,25.00,0.00,1.00



--- 📝 Object/Categorical Columns Summary ---


,_wandb,loss,tags,cache,logger,trainer,exp_name,ckpt_path,evaluator,hydra_cfg,optimizer,datamodule,datasource,transforms,profiler_cfg,subset_range,task_definition,experiment_group,plots/test/reconstructions_overview,plots/train/reconstructions_overview,plots/val/reconstructions_overview,model.kernels,model.strides,model._target_,model.dilations,model.output_channels,model.blocks_config,dataset._target_,dataset.dataset_cfg._target_,callbacks.model_timer._target_,callbacks.model_summary._target_,callbacks.early_stopping.mode,callbacks.early_stopping.monitor,callbacks.early_stopping._target_,callbacks.early_stopping.stopping_threshold,callbacks.early_stopping.divergence_threshold,callbacks.early_stopping.check_on_train_epoch_end,callbacks.model_checkpoint.mode,callbacks.model_checkpoint.dirpath,callbacks.model_checkpoint.monitor,callbacks.model_checkpoint._target_,callbacks.model_checkpoint.filename,callbacks.model_checkpoint.every_n_epochs,callbacks.model_checkpoint.every_n_train_steps,callbacks.model_checkpoint.train_time_interval,callbacks.model_checkpoint.save_on_train_epoch_end,callbacks.learning_rate_monitor._target_,callbacks.learning_rate_monitor.logging_interval,paths.log_dir,paths.ckpt_dir,paths.data_dir,paths.plot_dir,paths.root_dir,paths.work_dir,paths.cache_path,paths.output_dir,paths.eval_details,paths.artifacts_dir,paths.model_workdir,paths.model_cache_dir
count,54,54,54,54,54,54,54,0,54,54,54,54,54,54,0,0,54,54,48,50,50,54,54,54,54,54,2,54,54,54,54,54,54,54,0,0,0,54,54,54,54,54,0,0,0,0,54,54,54,54,54,54,54,54,54,54,54,54,54,54
unique,51,1,1,1,54,51,2,0,51,54,6,6,2,1,0,0,1,1,48,50,50,2,2,1,2,1,1,1,1,1,1,1,1,1,0,0,0,1,51,1,1,1,0,0,0,0,1,1,51,51,1,51,1,1,2,51,51,1,51,51
top,{'runtime': 1875},{'_target_': 'lmetk.loss.default.MSELoss'},[dev],"{'use_cache_after_loading': True, 'use_cache_a...",{'wandb': {'id': 'unibo+reconstruction+residua...,"{'devices': [1], '_target_': 'lightning.pytorc...",unibo/reconstruction/residual_autoencoder,NaN,{'val': {'paths': {'log_dir': '/home/levtel/pr...,"{'job': {'id': '???', 'num': '???', 'name': '$...",{'_target_': 'lmetk.optimizer.default.DefaultO...,{'_target_': 'lmetk.data.datamodules.base.Base...,{'_target_': 'lmetk.data.datasources.unibo.loa...,{'scaler_features': {'metadata': {'fit_on': 't...,NaN,NaN,{'model': {'data_requirements': {'input_tensor...,debug,"{'_type': 'plotly-file', 'path': 'media/plotly...","{'_type': 'plotly-file', 'path': 'media/plotly...","{'_type': 'plotly-file', 'path': 'media/plotly...","[5, 5, 5, 5, 5]","[1, 1, 1, 1, 1]",lmetk.model.autoencoder.wrapper.AutoencoderWra...,"[1, 2, 4, 8, 1]","[16, 32, 64, 64, 64]","[{'in': 2, 'out': 16, 'kernel': 7, 'stride': 2...",lmetk.data.datasets.hydra_concat_dataset.Hydra...,lmetk.data.datasets.sliding_window_batch_datas...,lmetk.callbacks.timer_callback.PipelineTimer,lightning.pytorch.callbacks.RichModelSummary,min,val/loss,lightning.pytorch.callbacks.EarlyStopping,NaN,NaN,NaN,min,/home/levtel/projects/TabPHM/artifacts/debug/r...,val/loss,lightning.pytorch.callbacks.ModelCheckpoint,epoch{epoch:03d}-val_loss{val/loss:.2f},NaN,NaN,NaN,NaN,lightning.pytorch.callbacks.LearningRateMonitor,epoch,/home/levtel/projects/TabPHM/artifacts/debug/r...,/home/levtel/projects/TabPHM/artifacts/debug/r...,/home/levtel/projects/TabPHM/datasets,/home/levtel/projects/TabPHM/artifacts/debug/r...,/home/levtel/projects/TabPHM,/home/levtel/projects/TabPHM,/home/levtel/projects/TabPHM/datasets/cache/UN...,/home/levtel/projects/TabPHM/artifacts/debug/r...,/home/levtel/projects/TabPHM/artifacts/debug/r...,/home/levtel/projects/TabPHM/artifacts,/home/levtel/projects/TabPHM/artifacts/debug/r...,/home/levtel/projects/TabPHM/artifacts/debug/r...
freq,3,54,54,54,1,2,52,NaN,2,1,14,14,52,54,NaN,NaN,54,54,1,1,1,27,27,54,27,54,2,54,54,54,54,54,54,54,NaN,NaN,NaN,54,2,54,54,54,NaN,NaN,NaN,NaN,54,54,2,2,54,2,54,54,52,2,2,54,2,2


### Remove columns that are not needed (we shouldn't vary them or their variation is not interesting)

In [9]:

remove_col = []
df = df.drop(remove_col, axis=1)

# Ensure that removed columns are not present in config_columns
config_columns = [col for col in config_columns if col != remove_col]

### Drop nan values 

In [ ]:
inspect_cols = []
for col in df.columns:
    # Check is the col
print(
    f"Number of rows with model.backbone._target_ = nan is {sum(df['model.backbone._target_'].isna())}"
)
# Drop na values if there are
df = df.dropna(subset=["model.backbone._target_"])
# Reset index
df = df.reset_index(drop=True)

# Drop rows that 'callbacks.early_stopping.monitor' isna
print(
    f"Number of rows with callbacks.early_stopping.monitor = nan is {sum(df['callbacks.early_stopping.monitor'].isna())}"
)



KeyError: 'model.backbone._target_'

In [34]:
df["model.backbone._target_"].unique()

array(['SANN'], dtype=object)

In [35]:
# Identify unique models in DataFrame
unique_models = df["model.backbone._target_"].unique()

# Identify unique datasets in DataFrame
unique_datasets = df["dataset.loader.parameters.data_name"].unique()


collected_results = defaultdict(dict)
collected_results_time = defaultdict(dict)
collected_results_time_run = defaultdict(dict)

collected_aggregated_results = defaultdict(dict)
collected_non_aggregated_results = defaultdict(dict)

# Got over each dataset and model and find the best result
for dataset in unique_datasets:
    for model in unique_models:
        # Get the subset of the DataFrame for the current dataset and model
        subset = df[
            (df["dataset.loader.parameters.data_name"] == dataset)
            & (df["model.backbone._target_"] == model)
        ]

        if subset.empty:
            print("---------")
            print(f"No results for {model} on {dataset}")
            print("---------")
            continue
        # Suppress all warnings
        warnings.filterwarnings("ignore")
        subset["Model"] = model
        warnings.filterwarnings("default")

        def get_metric(df):
            metric_ = df["callbacks.early_stopping.monitor"].unique()
            assert len(metric_) == 1, "There should be only one metric to optimize"
            metric = metric_[0]
            return metric.split("/")[-1]

        # Cols to get statistics later
        # TODO: log maximum validation value for optimized metric
        performance_cols = [f"test/{get_metric(subset)}"]

        # Get the unique values for each config column
        unique_colums_values = {}
        for col in config_columns:
            try:
                unique_colums_values[col] = subset[col].unique()
            except:
                print(f"Attention the columns: {col}, has issues with unique values")

        # Keep only those keys that have more than one unique value
        unique_colums_values = {
            k: v for k, v in unique_colums_values.items() if len(v) > 1
        }

        # Print the unique values for each config column

        print(f"Unique values for each config column for {model} on {dataset}:")
        for col, unique in unique_colums_values.items():
            print(f"{col}: {unique}")
            print()
        print("---------")

        # Check if "special colums" are not in unique_colums_values
        # For example dataset.parameters.data_seed should not be in aggregation columns
        # If it is, then we should remove it from the list
        special_columns = ["dataset.parameters.data_seed"]

        for col in special_columns:
            if col in unique_colums_values:
                unique_colums_values.pop(col)

        # Obtain the aggregation columns
        aggregation_columns = ["Model"] + list(unique_colums_values.keys())

        collected_non_aggregated_results[dataset][model] = {
            "df": subset.copy(),
            "aggregation_columns": aggregation_columns,
            "performance_cols": performance_cols,
        }

        # Aggregate the subset by the aggregation columns and get the best result for each group
        aggregated = subset.groupby(aggregation_columns).agg(
            {col: ["mean", "std"] for col in performance_cols}
        )

        # Go from MultiIndex to Index
        aggregated = aggregated.reset_index()

        assert (
            len(subset["callbacks.early_stopping.mode"].unique()) == 1
        ), "There should be only one mode for early stopping"
        # Identify the mode of the early stopping mode

        if subset["callbacks.early_stopping.mode"].unique()[0] == "max":
            ascending = False
            final_best_ = aggregated.sort_values(
                by=(f"test/{get_metric(subset)}", "mean"), ascending=ascending
            ).head(1)
            final_best_ = (final_best_ * 100).round(2)
        else:
            ascending = True
            final_best_ = aggregated.sort_values(
                by=(f"test/{get_metric(subset)}", "mean"), ascending=ascending
            ).head(1)

        collected_results[dataset][model] = {
            "mean": final_best_[(f"test/{get_metric(subset)}", "mean")].values[0],
            "std": final_best_[(f"test/{get_metric(subset)}", "std")].values[0],
        }

        # Get average epoch run time
        collected_results_time[dataset][model] = {
            "mean": subset['AvgTime/train_epoch_mean'].mean(),
            "std": subset['AvgTime/train_epoch_mean'].std(),
        }

        collected_results_time_run[dataset][model] = {
            "mean": subset["_runtime"].mean(),
            "std": subset["_runtime"].std(),
        }

        collected_aggregated_results[dataset][model] = aggregated.sort_values(
            by=(f"test/{get_metric(subset)}", "mean"), ascending=ascending
        )

Attention the columns: model.feature_encoder.in_channels, has issues with unique values
Attention the columns: model.feature_encoder.dataset_in_channels, has issues with unique values
Attention the columns: model.feature_encoder.selected_dimensions, has issues with unique values
Attention the columns: dataset.parameters.num_features, has issues with unique values
Attention the columns: callbacks.model_checkpoint.dirpath, has issues with unique values
Unique values for each config column for SANN on ogbg-molhiv:
model.readout.max_hop: [5. 4. 2.]

model.readout.hidden_dim: [128. 256.]

model.backbone.max_hop: [5. 4. 2.]

model.backbone.in_channels: [128. 256.]

model.backbone.hidden_channels: [128. 256.]

model.feature_encoder.max_hop: [5. 4. 2.]

model.feature_encoder.out_channels: [128. 256.]

model.feature_encoder.proj_dropout: [0.   0.25]

model.backbone_wrapper.max_hop: [5. 4. 2.]

model.backbone_wrapper.out_channels: [128. 256.]

dataset.split_params.data_seed: [0. 3. 5. 7. 9.]

pa

### Runtime per epoch

In [36]:
collected_results_time
# Convert nested dictionary to DataFrame
nested_dict = dict(collected_results_time)
result_dict = pd.DataFrame.from_dict(
    {
        (i, j): nested_dict[i][j]
        for i in nested_dict
        for j in nested_dict[i].keys()
    },
    orient="index",
)


result_dict = result_dict.round(2)
result_dict["performance"] = result_dict.apply(
    lambda x: f"{x['mean']} ± {x['std']}", axis=1
)
result_dict = result_dict.drop(["mean", "std"], axis=1)

# Reset multiindex
result_dict = result_dict.reset_index()
# rename columns
result_dict.columns = ["Dataset", "Model", "Average Time per Epoch"]

In [38]:
result_dict

,Dataset,Model,Average Time per Epoch
0,ogbg-molhiv,SANN,15.75 ± 2.78


In [39]:
result_dict.pivot_table(
    index="Model", columns="Dataset", values="Average Time per Epoch", aggfunc="first"
)[['ogbg-molhiv']] #[['MUTAG', 'NCI1','NCI109','PROTEINS','ZINC']]

Dataset,ogbg-molhiv
Model,
SANN,15.75 ± 2.78


### Runtime

In [40]:
collected_results_time_run
# Convert nested dictionary to DataFrame
nested_dict = dict(collected_results_time_run)
result_dict = pd.DataFrame.from_dict(
    {
        (i, j): nested_dict[i][j]
        for i in nested_dict
        for j in nested_dict[i].keys()
    },
    orient="index",
)


result_dict = result_dict.round(2)
result_dict["performance"] = result_dict.apply(
    lambda x: f"{x['mean']} ± {x['std']}", axis=1
)
result_dict = result_dict.drop(["mean", "std"], axis=1)

# Reset multiindex
result_dict = result_dict.reset_index()
# rename columns
result_dict.columns = ["Dataset", "Model", "Average Training Time"]


In [43]:
result_dict.pivot_table(
    index="Model", columns="Dataset", values="Average Training Time", aggfunc="first"
)[['ogbg-molhiv']] #[['MUTAG', 'NCI1','NCI109','PROTEINS','ZINC']]

Dataset,ogbg-molhiv
Model,
SANN,1378.66 ± 306.25
